In [2]:
import cv2
import numpy as np
import tensorflow as tf

# Load trained model
model = tf.keras.models.load_model("/content/mnist_model (1).h5")

frame_buffer = []

def preprocess(img):
    """MNIST-style preprocessing for webcam"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # Adaptive threshold + invert
    bw = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                               cv2.THRESH_BINARY_INV, 11, 2)
    # Morphological close to normalize stroke
    bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, np.ones((2,2), np.uint8))
    bw = cv2.medianBlur(bw, 3)
    return bw

def get_prediction(img):
    """Predict digit from preprocessed image"""
    contours, _ = cv2.findContours(img.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return -1, 0.0

    c = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(c)
    if w == 0 or h == 0:
        return -1, 0.0

    roi = img[y:y+h, x:x+w]
    h_r, w_r = roi.shape
    if h_r > w_r:
        new_h = 20
        new_w = max(int(w_r * 20 / h_r),1)
    else:
        new_w = 20
        new_h = max(int(h_r * 20 / w_r),1)

    resized = cv2.resize(roi, (new_w, new_h))
    padded = np.zeros((28,28), dtype=np.uint8)
    x_off = (28 - new_w)//2
    y_off = (28 - new_h)//2
    padded[y_off:y_off+new_h, x_off:x_off+new_w] = resized

    # Center using moments
    M = cv2.moments(padded)
    if M["m00"] != 0:
        cx = int(M["m10"]/M["m00"])
        cy = int(M["m01"]/M["m00"])
        shiftx = 14 - cx
        shifty = 14 - cy
        padded = cv2.warpAffine(padded, np.float32([[1,0,shiftx],[0,1,shifty]]), (28,28))

    # Normalize
    padded = padded.astype("float32") / 255.0
    padded = padded.reshape(1,28,28,1)

    prob = model.predict(padded, verbose=0)[0]
    return np.argmax(prob), np.max(prob)

# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w, _ = frame.shape
    size = 300
    x1, y1 = w//2 - size//2, h//2 - size//2
    x2, y2 = x1 + size, y1 + size

    roi = frame[y1:y2, x1:x2]
    processed = preprocess(roi)
    digit, conf = get_prediction(processed)

    # Smooth over last 5 frames
    frame_buffer.append((digit, conf))
    if len(frame_buffer) > 5:
        frame_buffer.pop(0)

    counts = np.zeros(10)
    for d,c in frame_buffer:
        if d != -1:
            counts[d] += c
    if counts.sum() > 0:
        digit = counts.argmax()
        conf = counts[digit]/counts.sum()
    else:
        digit = -1
        conf = 0

    # Display
    cv2.rectangle(frame, (x1,y1), (x2,y2), (255,255,0), 2)
    text = f"Digit: {digit} ({conf*100:.1f}%)" if digit!=-1 else "No digit"
    cv2.putText(frame, text, (10,40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,200,0),2)

    cv2.imshow("Digit Recognition", frame)
    cv2.imshow("Processed ROI", processed)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
